# Wahl-O-Mat Feature Generation All Time Projection

In [21]:
%load_ext autoreload
%autoreload 2
import helper
import plotting
import constants
import pandas as pd
from datetime import datetime
import bisect
import numpy as np
import ast
import matplotlib.pyplot as plt
import pm4py

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
TOP_K = 3

In [23]:
# what should be available at this point:
# Event logs including embedding column
# Wahl-o-mat topic embeddings
# Wahl-o-mat agreement scores per topic

In [24]:
# now the idea:
# for each trace, find the closest topic via similarity of embeddings
# store the similarity score as a case attribute in the dataframe
# pick the correct aggrement scores on the topic and store them also as case attributes

## Berlin

In [25]:
berlin_df_allTime = pd.read_csv("./data-csv/Berlin-with-embeddings-allTime.csv")
berlin_df_allTime['case:embedding'] = berlin_df_allTime['case:embedding'].apply(lambda x: np.array(ast.literal_eval(x)))
berlin_wahlOmat_embeddings_2023 = pd.read_csv("./wahl-o-mat-topic-embeddings/topic-embeddings-berlin-2023.csv")
berlin_wahlOmat_embeddings_2023['embedding'] = berlin_wahlOmat_embeddings_2023['embedding'].apply(lambda x: np.array(ast.literal_eval(x)))
berlin_wahlOmat_embeddings = berlin_wahlOmat_embeddings_2023

# Preload all CSVs into a dict keyed by start date
BERLIN_AGREEMENT_SCORES = {
    d: pd.read_csv(f"./wahl-o-mat-agreement-scores/{v['agreement_score_file']}")
    for d, v in constants.BERLIN_COALITIONS_SORTED
}

def get_berlin_agreement_scores(dateStr):
    query_date = datetime.strptime(dateStr, "%Y-%m-%d %H:%M:%S%z").replace(tzinfo=None)

    # Find the most recent coalition start date ≤ query_date
    idx = bisect.bisect_right(
        [d for d, _ in constants.BERLIN_COALITIONS_SORTED],
        query_date
    ) - 1

    if idx < 0:
        print(f"No Berlin coalition defined before {query_date} - using earliest available agreement scores")
        idx = 0

    start_date = constants.BERLIN_COALITIONS_SORTED[idx][0]

    # Return the preloaded DataFrame
    return BERLIN_AGREEMENT_SCORES[start_date]

In [ ]:
# normalize topic embeddings (so that cosine similarity is only dot product then):
T = np.vstack(berlin_wahlOmat_embeddings['embedding'].values)
T_norm = T / np.clip(np.linalg.norm(T, axis=1, keepdims=True), 1e-10, None)

results = []

trace_df = berlin_df_allTime.groupby('case:concept:name').first().reset_index()

for i, row in trace_df.iterrows():
    event_emb = row["case:embedding"]

    top_topics = helper.get_top_k_for_event(event_emb, T_norm, berlin_wahlOmat_embeddings, top_k=TOP_K)
    results.append({
        "case:concept:name": row["case:concept:name"],
        "top_topics": top_topics,
        "top_topic": top_topics[0][0] if top_topics else None,
        "top_topic_similarity_score": top_topics[0][1] if top_topics else None
    })

# Convert to DataFrame
top_topics_df = pd.DataFrame(results) # has columns "case:concept:name" and "top_topics" (list of tuples with topic no. and score)

# now we have for each case:concept:name the topic no. of the top_k most similar topics
# this we can merge with the original dataframe and then also add the agreement score for the respective topic (for k = 1, for k > 1, we would have to think about it and aggregate something)

merged_berlin_df = berlin_df_allTime.merge(top_topics_df, on="case:concept:name", how="left")

# add agreement score for each topic in top_topics column
def add_agreement_score(row, key):
    if row["top_topics"]:
        topic_no, score = row["top_topics"][0]  # get the first (and only) topic tuple

        berlin_agreement_scores = get_berlin_agreement_scores(row["DokDat"])
        # toDo: see which agreement scores we need (2021 or 2023):
        agreement_score = berlin_agreement_scores.loc[berlin_agreement_scores['statement_number'] == topic_no, key].iloc[0]
        return agreement_score
    return None

merged_berlin_df["government_agreement_score"] = merged_berlin_df.apply(lambda row: add_agreement_score(row, "government"), axis=1)
merged_berlin_df["opposition_agreement_score"] = merged_berlin_df.apply(lambda row: add_agreement_score(row, "opposition"), axis=1)
merged_berlin_df["all_agreement_score"] = merged_berlin_df.apply(lambda row: add_agreement_score(row, "all"), axis=1)

merged_berlin_df.to_csv("./data-csv/Berlin-with-topics-and-agreement-scores-all-time.csv", index=False)
# use pm4py to create XES file
event_log_berlin = pm4py.format_dataframe(merged_berlin_df, case_id='case:concept:name', activity_key='DokTypL', timestamp_key='DokDat')
event_log_berlin_threshold_topic_similarity = event_log_berlin[
        event_log_berlin["top_topic_similarity_score"].between(
            0.5,
            1,
            inclusive="both"
        )
    ].copy()

pm4py.write_xes(event_log_berlin, "./data-xes/Berlin-with-topics-and-agreement-scores-all-time.xes")
pm4py.write_xes(event_log_berlin_threshold_topic_similarity, "./data-xes/Berlin-with-topics-and-agreement-scores-all-time-thresholded.xes")

No Berlin coalition defined before 1989-05-24 00:00:00 - using earliest available agreement scores
No Berlin coalition defined before 1989-06-01 00:00:00 - using earliest available agreement scores
No Berlin coalition defined before 1989-06-14 00:00:00 - using earliest available agreement scores
No Berlin coalition defined before 1989-06-14 00:00:00 - using earliest available agreement scores
No Berlin coalition defined before 1989-09-14 00:00:00 - using earliest available agreement scores
No Berlin coalition defined before 1989-05-24 00:00:00 - using earliest available agreement scores
No Berlin coalition defined before 1989-06-01 00:00:00 - using earliest available agreement scores
No Berlin coalition defined before 1989-09-11 00:00:00 - using earliest available agreement scores
No Berlin coalition defined before 1989-10-19 00:00:00 - using earliest available agreement scores
No Berlin coalition defined before 1989-10-19 00:00:00 - using earliest available agreement scores
No Berlin 

exporting log, completed traces ::   0%|          | 0/1813 [00:00<?, ?it/s]

exporting log, completed traces ::   0%|          | 0/329 [00:00<?, ?it/s]

## Baden-Württemberg

In [27]:
bawue_allTime_df = pd.read_csv("./data-csv/Baden-Württemberg-with-embeddings-allTime.csv")
bawue_allTime_df['case:embedding'] = bawue_allTime_df['case:embedding'].apply(lambda x: np.array(ast.literal_eval(x)))

bawue_wahlOmat_embeddings_2021 = pd.read_csv("./wahl-o-mat-topic-embeddings/topic-embeddings-bawue-2021.csv")
bawue_wahlOmat_embeddings_2021['embedding'] = bawue_wahlOmat_embeddings_2021['embedding'].apply(lambda x: np.array(ast.literal_eval(x)))

bawue_wahlOmat_embeddings = bawue_wahlOmat_embeddings_2021

# Preload all CSVs into a dict keyed by start date
BAWUE_AGREEMENT_SCORES = {
    d: pd.read_csv(f"./wahl-o-mat-agreement-scores/{v['agreement_score_file']}")
    for d, v in constants.BADEN_WUERTTEMBERG_COALITIONS_SORTED
}

def get_bawue_agreement_scores(dateStr):
    query_date = datetime.strptime(dateStr, "%Y-%m-%d %H:%M:%S%z").replace(tzinfo=None)

    # Find the most recent coalition start date ≤ query_date
    idx = bisect.bisect_right(
        [d for d, _ in constants.BADEN_WUERTTEMBERG_COALITIONS_SORTED],
        query_date
    ) - 1

    if idx < 0:
        print(f"No Baden-Württemberg coalition defined before {query_date} - using earliest available agreement scores")
        idx = 0

    start_date = constants.BADEN_WUERTTEMBERG_COALITIONS_SORTED[idx][0]

    # Return the preloaded DataFrame
    return BAWUE_AGREEMENT_SCORES[start_date]

In [28]:
# normalize topic embeddings (so that cosine similarity is only dot product then):
T = np.vstack(bawue_wahlOmat_embeddings['embedding'].values)
T_norm = T / np.clip(np.linalg.norm(T, axis=1, keepdims=True), 1e-10, None)

results = []

trace_df = bawue_allTime_df.groupby('case:concept:name').first().reset_index()

for i, row in trace_df.iterrows():
    event_emb = row["case:embedding"]

    top_topics = helper.get_top_k_for_event(event_emb, T_norm, bawue_wahlOmat_embeddings, top_k=TOP_K)
    results.append({
        "case:concept:name": row["case:concept:name"],
        "top_topics": top_topics,
        "top_topic": top_topics[0][0] if top_topics else None,
        "top_topic_similarity_score": top_topics[0][1] if top_topics else None
    })

# Convert to DataFrame
top_topics_df = pd.DataFrame(results) # has columns "case:concept:name" and "top_topics" (list of tuples with topic no. and score)

# now we have for each case:concept:name the topic no. of the top_k most similar topics
# this we can merge with the original dataframe and then also add the agreement score for the respective topic (for k = 1, for k > 1, we would have to think about it and aggregate something)

merged_bawue_df = bawue_allTime_df.merge(top_topics_df, on="case:concept:name", how="left")

# add agreement score for each topic in top_topics column
def add_agreement_score(row, key):
    if row["top_topics"]:
        topic_no, score = row["top_topics"][0]  # get the first (and only) topic tuple

        bawue_agreement_scores = get_bawue_agreement_scores(row["DokDat"])
        # toDo: see which agreement scores we need (2021 or 2023):
        agreement_score = bawue_agreement_scores.loc[bawue_agreement_scores['statement_number'] == topic_no, key].iloc[0]
        return agreement_score
    return None

merged_bawue_df["government_agreement_score"] = merged_bawue_df.apply(lambda row: add_agreement_score(row, "government"), axis=1)
merged_bawue_df["opposition_agreement_score"] = merged_bawue_df.apply(lambda row: add_agreement_score(row, "opposition"), axis=1)
merged_bawue_df["all_agreement_score"] = merged_bawue_df.apply(lambda row: add_agreement_score(row, "all"), axis=1)

merged_bawue_df.to_csv("./data-csv/Baden-Württemberg-with-topics-and-agreement-scores-all-time.csv", index=False)
# use pm4py to create XES file
event_log_bawue = pm4py.format_dataframe(merged_bawue_df, case_id='case:concept:name', activity_key='DokTypL', timestamp_key='DokDat')
event_log_bawue_threshold_topic_similarity = event_log_bawue[
        event_log_bawue["top_topic_similarity_score"].between(
            0.5,
            1,
            inclusive="both"
        )
    ].copy()
pm4py.write_xes(event_log_bawue, "./data-xes/Baden-Württemberg-with-topics-and-agreement-scores-all-time.xes")
pm4py.write_xes(event_log_bawue_threshold_topic_similarity, "./data-xes/Baden-Württemberg-with-topics-and-agreement-scores-all-time-thresholded.xes")

# ToDo: do further checking if all of this works as expected, e.g. check if the correct agreement scores are assigned for some cases, check the distribution of similarity scores, etc.


exporting log, completed traces ::   0%|          | 0/1465 [00:00<?, ?it/s]

exporting log, completed traces ::   0%|          | 0/268 [00:00<?, ?it/s]

## Brandenburg

In [29]:
brandenburg_allTime_df = pd.read_csv("./data-csv/Brandenburg-with-embeddings-allTime.csv")
brandenburg_allTime_df['case:embedding'] = brandenburg_allTime_df['case:embedding'].apply(lambda x: np.array(ast.literal_eval(x)))

brandenburg_wahlOmat_embeddings_2024 = pd.read_csv("./wahl-o-mat-topic-embeddings/topic-embeddings-brandenburg-2024.csv")
brandenburg_wahlOmat_embeddings_2024['embedding'] = brandenburg_wahlOmat_embeddings_2024['embedding'].apply(lambda x: np.array(ast.literal_eval(x)))

brandenburg_wahlOmat_embeddings = brandenburg_wahlOmat_embeddings_2024

# Preload all CSVs into a dict keyed by start date
BRANDENBURG_AGREEMENT_SCORES = {
    d: pd.read_csv(f"./wahl-o-mat-agreement-scores/{v['agreement_score_file']}")
    for d, v in constants.BRANDENBURG_COALITIONS_SORTED
}

def get_brandenburg_agreement_scores(dateStr):
    query_date = datetime.strptime(dateStr, "%Y-%m-%d %H:%M:%S%z").replace(tzinfo=None)

    # Find the most recent coalition start date ≤ query_date
    idx = bisect.bisect_right(
        [d for d, _ in constants.BRANDENBURG_COALITIONS_SORTED],
        query_date
    ) - 1

    if idx < 0:
        print(f"No Brandenburg coalition defined before {query_date} - using earliest available agreement scores")
        idx = 0

    start_date = constants.BRANDENBURG_COALITIONS_SORTED[idx][0]

    # Return the preloaded DataFrame
    return BRANDENBURG_AGREEMENT_SCORES[start_date]

In [ ]:
# normalize topic embeddings (so that cosine similarity is only dot product then):
T = np.vstack(brandenburg_wahlOmat_embeddings['embedding'].values)
T_norm = T / np.clip(np.linalg.norm(T, axis=1, keepdims=True), 1e-10, None)

results = []

trace_df = brandenburg_allTime_df.groupby('case:concept:name').first().reset_index()

for i, row in trace_df.iterrows():
    event_emb = row["case:embedding"]

    top_topics = helper.get_top_k_for_event(event_emb, T_norm, brandenburg_wahlOmat_embeddings, top_k=TOP_K)
    results.append({
        "case:concept:name": row["case:concept:name"],
        "top_topics": top_topics,
        "top_topic": top_topics[0][0] if top_topics else None,
        "top_topic_similarity_score": top_topics[0][1] if top_topics else None
    })

# Convert to DataFrame
top_topics_df = pd.DataFrame(results) # has columns "case:concept:name" and "top_topics" (list of tuples with topic no. and score)

# now we have for each case:concept:name the topic no. of the top_k most similar topics
# this we can merge with the original dataframe and then also add the agreement score for the respective topic (for k = 1, for k > 1, we would have to think about it and aggregate something)

merged_brandenburg_df = brandenburg_allTime_df.merge(top_topics_df, on="case:concept:name", how="left")

# add agreement score for each topic in top_topics column
def add_agreement_score(row, key):
    if row["top_topics"]:
        topic_no, score = row["top_topics"][0]  # get the first (and only) topic tuple

        brandenburg_agreement_scores = get_brandenburg_agreement_scores(row["DokDat"])
        # toDo: see which agreement scores we need (2021 or 2023):
        agreement_score = brandenburg_agreement_scores.loc[brandenburg_agreement_scores['statement_number'] == topic_no, key].iloc[0]
        return agreement_score
    return None

merged_brandenburg_df["government_agreement_score"] = merged_brandenburg_df.apply(lambda row: add_agreement_score(row, "government"), axis=1)
merged_brandenburg_df["opposition_agreement_score"] = merged_brandenburg_df.apply(lambda row: add_agreement_score(row, "opposition"), axis=1)
merged_brandenburg_df["all_agreement_score"] = merged_brandenburg_df.apply(lambda row: add_agreement_score(row, "all"), axis=1)

merged_brandenburg_df.to_csv("./data-csv/Brandenburg-with-topics-and-agreement-scores-all-time.csv", index=False)
# use pm4py to create XES file
event_log_brandenburg = pm4py.format_dataframe(merged_brandenburg_df, case_id='case:concept:name', activity_key='DokTypL', timestamp_key='DokDat')
event_log_brandenburg_threshold_topic_similarity = event_log_brandenburg[
        event_log_brandenburg["top_topic_similarity_score"].between(
            0.5,
            1,
            inclusive="both"
        )
    ].copy()
pm4py.write_xes(event_log_brandenburg, "./data-xes/Brandenburg-with-topics-and-agreement-scores-all-time.xes")
pm4py.write_xes(event_log_brandenburg_threshold_topic_similarity, "./data-xes/Brandenburg-with-topics-and-agreement-scores-all-time-thresholded.xes")


exporting log, completed traces ::   0%|          | 0/1410 [00:00<?, ?it/s]

exporting log, completed traces ::   0%|          | 0/264 [00:00<?, ?it/s]